In [0]:
RUNDIR = 'S25A_April2026'
RUN = RUNDIR.replace('/', '_')
ARM = 'b'
SPECTROGRAPH = '1'
DATE = '20??????'
VISIT = '??????'
PFSARM_PATH = f'/scratch/aszalay1/dobos/pfs/data/repo/ssp/S25A-OT02/2d/{RUNDIR}/pfsArm/{DATE}/{VISIT}/pfsArm_PFS_{VISIT}_{ARM}{SPECTROGRAPH}_{RUN}.fits'
OBSLOG_PATH = '/home/dobos/project/Subaru-PFS/spt_ssp_observation/runs/2025-06/obslog/*.csv'
OBSLOG_PATTERN = 'SSP_GA_dra_S0P00'

In [0]:
import os
import re
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits

In [0]:
visit = []

for path in glob(OBSLOG_PATH):
    with open(path) as f:
        for line in f:
            if OBSLOG_PATTERN in line:
                first_field = line.split(",", 1)[0]
                visit.append(first_field)

len(visit), print(visit)

In [0]:
files = {}

for path in glob(PFSARM_PATH):
    m = re.search(rf'pfsArm_PFS_(\d+)_({ARM}{SPECTROGRAPH})_{RUN}.fits', path)
    if m and m.group(1) in visit:
        v = m.group(1)
        files[v] = path

len(files), files

In [0]:
from pfs.datamodel import PfsArm

In [0]:
y = np.arange(4176, dtype=float)
wave = []
fiberid = []

for v, path in files.items():
    arm = PfsArm.readFits(path)
    wave.append(arm.wavelength)
    fiberid.append(arm.fiberId)

wave = np.stack(wave)
fiberid = np.stack(fiberid)

wave.shape, fiberid.shape

In [0]:
for w in wave:
    print(w.shape)

In [0]:
wave.shape

In [0]:
fig, ax = plt.subplots(figsize=(12, 4), dpi=120)

cmap = plt.get_cmap('tab20', wave.shape[0])

for i in range(wave.shape[0]):                  # visits
    for j in range(0, wave.shape[1], 100):      # fibers
        ax.plot(y, wave[i, j] - wave[0, j], '.', ms=0.2, c=cmap(i))

ax.set_xlabel('y [pixel]')
ax.set_ylabel(R'$\Delta \lambda$ [nm]')

ax.set_xlim(0, 4176)

ax2 = ax.twiny()
ax2.set_xlabel(R'$\lambda$ [nm] (approximate)')
ax2.set_xlim(wave[0, 0, 0], wave[0, 0, -1])

ax.set_title('Wavelength difference from first visit\n' +
              f'RUN={RUN}, ARM={ARM}, SPECTROGRAPH={SPECTROGRAPH}\n' +
              f'VISITS={",".join(visit)}',
              fontsize=8)